# The Self-Pruning Neural Network
**Tredence Analytics — AI Engineering Internship 2025**

A neural network that learns to prune itself during training using learnable gates and L1 regularization on CIFAR-10.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import math
import copy
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
os.makedirs('output', exist_ok=True)

Using device: cuda


## Part 1: The PrunableLinear Layer

Custom linear layer that cannot use `nn.Linear`. Each weight has a corresponding learnable `gate_score`.

**Forward pass:** `output = x @ (weight * sigmoid(gate_scores))^T + bias`

Gradients flow through both `weight` and `gate_scores` via PyTorch autograd.

In [2]:
class PrunableLinear(nn.Module):
    """Custom linear layer with learnable gate parameters for self-pruning.
    
    Each weight w_ij has a corresponding gate_score g_ij.
    The effective weight is: w_ij * sigmoid(g_ij)
    When sigmoid(g_ij) -> 0, the weight is effectively pruned.
    """
    
    def __init__(self, in_features, out_features):
        super(PrunableLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        # Standard weight and bias parameters
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.empty(out_features))
        
        # Learnable gate scores — same shape as weight
        self.gate_scores = nn.Parameter(torch.empty(out_features, in_features))
        
        self.reset_parameters()
    
    def reset_parameters(self):
        # Kaiming initialization for weights (standard for ReLU networks)
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
        bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
        nn.init.uniform_(self.bias, -bound, bound)
        
        # Initialize gate_scores to 1.0 so sigmoid(1.0) ~ 0.73
        # All gates start mostly open, allowing full gradient flow.
        nn.init.constant_(self.gate_scores, 1.0)
    
    def forward(self, x):
        # Step 1: Transform gate_scores to (0,1) range via sigmoid
        gates = torch.sigmoid(self.gate_scores)
        # Step 2: Element-wise multiplication creates pruned weights
        pruned_weights = self.weight * gates
        # Step 3: Standard linear operation: output = x @ pruned_weights^T + bias
        return F.linear(x, pruned_weights, self.bias)

# Verify gradient flow
print('--- Gradient Flow Sanity Check ---')
test_layer = PrunableLinear(4, 2).to(device)
test_x = torch.randn(1, 4, device=device)
test_out = test_layer(test_x).sum()
test_out.backward()
print(f'  weight.grad exists: {test_layer.weight.grad is not None}')
print(f'  gate_scores.grad exists: {test_layer.gate_scores.grad is not None}')
print(f'  bias.grad exists: {test_layer.bias.grad is not None}')
print('  All gradients flow correctly through both weight and gate_scores!')

--- Gradient Flow Sanity Check ---
  weight.grad exists: True
  gate_scores.grad exists: True
  bias.grad exists: True
  All gradients flow correctly through both weight and gate_scores!


## Part 2: Network Architecture

Standard feed-forward MLP for CIFAR-10 classification:

```
Input (3x32x32 = 3072) -> PrunableLinear(3072, 512) -> ReLU
                        -> PrunableLinear(512, 256)  -> ReLU
                        -> PrunableLinear(256, 10)   -> Output
```

In [3]:
class SelfPruningNet(nn.Module):
    """3-layer MLP using PrunableLinear layers."""
    
    def __init__(self):
        super(SelfPruningNet, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = PrunableLinear(3 * 32 * 32, 512)
        self.fc2 = PrunableLinear(512, 256)
        self.fc3 = PrunableLinear(256, 10)
    
    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    
    def prunable_layers(self):
        return [self.fc1, self.fc2, self.fc3]
    
    def get_gate_params(self):
        """Return only gate_scores parameters."""
        return [layer.gate_scores for layer in self.prunable_layers()]
    
    def get_non_gate_params(self):
        """Return all parameters EXCEPT gate_scores."""
        gate_ids = {id(p) for p in self.get_gate_params()}
        return [p for p in self.parameters() if id(p) not in gate_ids]

# Print architecture info
model = SelfPruningNet()
total_gates = sum(l.gate_scores.numel() for l in model.prunable_layers())
print(f'\nArchitecture: 3072 -> 512 -> 256 -> 10')
print(f'Total gate parameters: {total_gates:,}')
for i, l in enumerate(model.prunable_layers(), 1):
    print(f'  FC{i}: {l.in_features} x {l.out_features} = {l.gate_scores.numel():,} gates')
del model


Architecture: 3072 -> 512 -> 256 -> 10
Total gate parameters: 1,706,496
  FC1: 3072 x 512 = 1,572,864 gates
  FC2: 512 x 256 = 131,072 gates
  FC3: 256 x 10 = 2,560 gates


## Part 3: Sparsity Loss and Metrics

**Sparsity Loss** = L1 norm of all gates = `sum(sigmoid(gate_scores))` across all layers.

Since gates are always positive after sigmoid, the L1 norm is simply their sum.

**Total Loss** = `CrossEntropyLoss + lambda * SparsityLoss`

In [4]:
def compute_sparsity_loss(model):
    """L1 norm of all gate values = sum of sigmoid(gate_scores).
    Since gates are always positive after sigmoid, |g| = g."""
    total = torch.tensor(0.0, device=device)
    for layer in model.prunable_layers():
        gates = torch.sigmoid(layer.gate_scores)
        total = total + gates.sum()
    return total

def compute_sparsity_level(model, threshold=1e-2):
    """Percentage of gates below threshold (effectively pruned)."""
    pruned = 0
    total = 0
    with torch.no_grad():
        for layer in model.prunable_layers():
            gates = torch.sigmoid(layer.gate_scores)
            pruned += (gates < threshold).sum().item()
            total += gates.numel()
    return (pruned / total) * 100.0

def get_all_gate_values(model):
    """Collect all gate values as a flat numpy array for plotting."""
    vals = []
    with torch.no_grad():
        for layer in model.prunable_layers():
            g = torch.sigmoid(layer.gate_scores).cpu().numpy().flatten()
            vals.extend(g)
    return np.array(vals)

## Part 4: Training Loop

**Key design decision:** We use **two separate optimizers** with different learning rates:
- **Adam (lr=1e-3)** for weights and biases — standard classification learning
- **Adam (lr=0.05)** for gate_scores — a higher learning rate ensures the constant L1 sparsity gradient can overcome Adam's adaptive normalization and actually drive gates to zero

This is necessary because the sigmoid's vanishing derivative near 0 makes it very hard for gates to cross the pruning threshold (0.01) at standard learning rates.

In [5]:
def train_model(lambda_val, epochs=20):
    """Train a self-pruning network with the given lambda."""
    print(f'\n{"="*60}')
    print(f'  Training with lambda = {lambda_val}')
    print(f'{"="*60}')
    
    # Data preparation
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010))
    ])
    train_set = datasets.CIFAR10('./data', train=True,  download=True, transform=transform)
    test_set  = datasets.CIFAR10('./data', train=False, download=True, transform=transform)
    train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2)
    test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2)
    
    # Model
    model = SelfPruningNet().to(device)
    criterion = nn.CrossEntropyLoss()
    
    # Two separate optimizers with different learning rates
    opt_weights = optim.Adam(model.get_non_gate_params(), lr=1e-3)
    opt_gates   = optim.Adam(model.get_gate_params(), lr=0.05)
    
    history = {'train_loss': [], 'clf_loss': [], 'test_acc': [], 'sparsity': []}
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        running_clf = 0.0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            opt_weights.zero_grad()
            opt_gates.zero_grad()
            
            outputs = model(inputs)
            clf_loss = criterion(outputs, labels)
            sparsity_loss = compute_sparsity_loss(model)
            
            # Total Loss = Classification Loss + lambda * Sparsity Loss
            total_loss = clf_loss + lambda_val * sparsity_loss
            total_loss.backward()
            
            opt_weights.step()
            opt_gates.step()
            
            running_loss += total_loss.item()
            running_clf += clf_loss.item()
        
        # Evaluate
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                preds = model(inputs).argmax(dim=1)
                correct += preds.eq(labels).sum().item()
                total += labels.size(0)
        
        test_acc = 100. * correct / total
        sparsity = compute_sparsity_level(model)
        avg_loss = running_loss / len(train_loader)
        avg_clf = running_clf / len(train_loader)
        
        history['train_loss'].append(avg_loss)
        history['clf_loss'].append(avg_clf)
        history['test_acc'].append(test_acc)
        history['sparsity'].append(sparsity)
        
        print(f'Epoch {epoch+1:02d}/{epochs} | CLF Loss: {avg_clf:.4f} | '
              f'Test Acc: {test_acc:.2f}% | Sparsity: {sparsity:.2f}%')
    
    return model, history

## Part 5: Run Experiments
We sweep lambda across 5 values: `[1e-5, 1e-4, 5e-4, 1e-3, 2e-3]`

In [6]:
EPOCHS = 20
lambdas = [1e-5, 1e-4, 5e-4, 1e-3, 2e-3]

all_results = {}
all_models = {}

for lam in lambdas:
    model, history = train_model(lam, epochs=EPOCHS)
    all_results[lam] = history
    all_models[lam] = copy.deepcopy(model.state_dict())


  Training with lambda = 1e-05


100%|██████████| 170M/170M [00:23<00:00, 7.26MB/s] 


Epoch 01/20 | CLF Loss: 1.6499 | Test Acc: 47.08% | Sparsity: 69.86%
Epoch 02/20 | CLF Loss: 1.4445 | Test Acc: 49.64% | Sparsity: 91.03%
Epoch 03/20 | CLF Loss: 1.3590 | Test Acc: 51.48% | Sparsity: 94.77%
Epoch 04/20 | CLF Loss: 1.3043 | Test Acc: 52.49% | Sparsity: 96.51%
Epoch 05/20 | CLF Loss: 1.2653 | Test Acc: 52.89% | Sparsity: 97.38%
Epoch 06/20 | CLF Loss: 1.2280 | Test Acc: 53.32% | Sparsity: 97.85%
Epoch 07/20 | CLF Loss: 1.1993 | Test Acc: 52.84% | Sparsity: 98.14%
Epoch 08/20 | CLF Loss: 1.1745 | Test Acc: 53.61% | Sparsity: 98.34%
Epoch 09/20 | CLF Loss: 1.1550 | Test Acc: 53.69% | Sparsity: 98.48%
Epoch 10/20 | CLF Loss: 1.1356 | Test Acc: 54.12% | Sparsity: 98.60%
Epoch 11/20 | CLF Loss: 1.1182 | Test Acc: 53.92% | Sparsity: 98.69%
Epoch 12/20 | CLF Loss: 1.1020 | Test Acc: 54.06% | Sparsity: 98.76%
Epoch 13/20 | CLF Loss: 1.0853 | Test Acc: 54.43% | Sparsity: 98.82%
Epoch 14/20 | CLF Loss: 1.0718 | Test Acc: 53.89% | Sparsity: 98.87%
Epoch 15/20 | CLF Loss: 1.0580 | T

## Part 6: Results and Visualizations

In [7]:
# Summary Table
print('\n' + '='*55)
print(f'{"Lambda":<10} | {"Test Acc (%)":<14} | {"Sparsity (%)":<14}')
print('-'*55)
for lam in lambdas:
    acc = all_results[lam]['test_acc'][-1]
    sp = all_results[lam]['sparsity'][-1]
    print(f'{lam:<10.0e} | {acc:<14.2f} | {sp:<14.2f}')
print('='*55)


Lambda     | Test Acc (%)   | Sparsity (%)  
-------------------------------------------------------
1e-05      | 54.29          | 99.04         
1e-04      | 47.37          | 99.83         
5e-04      | 41.10          | 99.93         
1e-03      | 33.43          | 99.97         
2e-03      | 10.00          | 100.00        


In [8]:
# Plot 1: Training curves — Test Accuracy and Sparsity over epochs
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for lam in lambdas:
    label = f'\u03bb={lam:.0e}'
    ax1.plot(range(1, EPOCHS+1), all_results[lam]['test_acc'], label=label)
    ax2.plot(range(1, EPOCHS+1), all_results[lam]['sparsity'], label=label)

ax1.set_title('Test Accuracy over Epochs', fontsize=13)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy (%)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_title('Sparsity Level over Epochs', fontsize=13)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Sparsity (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('output/training_curves.png', dpi=150)
plt.show()

In [9]:
# Plot 2: Sparsity vs Accuracy trade-off scatter
fig, ax = plt.subplots(figsize=(8, 5))
accs = [all_results[l]['test_acc'][-1] for l in lambdas]
spars = [all_results[l]['sparsity'][-1] for l in lambdas]

ax.plot(spars, accs, 'ro-', markersize=10, linewidth=2)
for i, l in enumerate(lambdas):
    ax.annotate(f'\u03bb={l:.0e}', (spars[i], accs[i]),
               textcoords='offset points', xytext=(0, 12), ha='center', fontsize=9)

ax.set_title('Sparsity vs Test Accuracy Trade-off', fontsize=13)
ax.set_xlabel('Sparsity Level (%)')
ax.set_ylabel('Test Accuracy (%)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/sparsity_vs_accuracy.png', dpi=150)
plt.show()

In [10]:
# Plot 3: Gate value distributions for all lambdas (combined figure)
fig, axes = plt.subplots(1, len(lambdas), figsize=(4*len(lambdas), 4))

for i, lam in enumerate(lambdas):
    m = SelfPruningNet().to(device)
    m.load_state_dict(all_models[lam])
    gates = get_all_gate_values(m)
    sp = all_results[lam]['sparsity'][-1]
    ac = all_results[lam]['test_acc'][-1]
    
    axes[i].hist(gates, bins=50, color='teal', edgecolor='black', alpha=0.7)
    axes[i].set_title(f'\u03bb={lam:.0e}\nSp={sp:.1f}% Acc={ac:.1f}%', fontsize=10)
    axes[i].set_xlabel('Gate Value')
    axes[i].set_ylabel('Count')
    axes[i].set_yscale('log')

plt.suptitle('Gate Value Distributions Across Lambda Values', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('output/gate_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [11]:
# Plot 4: Per-lambda detailed plot (gate histogram + training progress)
for lam in lambdas:
    m = SelfPruningNet().to(device)
    m.load_state_dict(all_models[lam])
    gates = get_all_gate_values(m)
    sp = all_results[lam]['sparsity'][-1]
    ac = all_results[lam]['test_acc'][-1]
    
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Left: Gate histogram
    a1.hist(gates, bins=50, color='teal', edgecolor='black', alpha=0.7)
    a1.set_title(f'Gate Distribution (\u03bb={lam:.0e})')
    a1.set_xlabel('Gate Value (after Sigmoid)')
    a1.set_ylabel('Frequency')
    a1.set_yscale('log')
    a1.grid(True, alpha=0.3)
    
    # Right: Accuracy + Sparsity over epochs (dual y-axis)
    ep = range(1, EPOCHS+1)
    c1 = 'tab:blue'
    a2.plot(ep, all_results[lam]['test_acc'], color=c1, label='Test Accuracy')
    a2.set_ylabel('Accuracy (%)', color=c1)
    a2.tick_params(axis='y', labelcolor=c1)
    a2.set_xlabel('Epoch')
    
    a2_twin = a2.twinx()
    c2 = 'tab:red'
    a2_twin.plot(ep, all_results[lam]['sparsity'], color=c2, label='Sparsity')
    a2_twin.set_ylabel('Sparsity (%)', color=c2)
    a2_twin.tick_params(axis='y', labelcolor=c2)
    a2.set_title(f'Training Progress (\u03bb={lam:.0e})')
    a2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'output/plot_lambda_{lam}.png', dpi=150)
    plt.show()
    print(f'\u03bb={lam:.0e} | Sparsity: {sp:.2f}% | Accuracy: {ac:.2f}%')

λ=1e-05 | Sparsity: 99.04% | Accuracy: 54.29%
λ=1e-04 | Sparsity: 99.83% | Accuracy: 47.37%
λ=5e-04 | Sparsity: 99.93% | Accuracy: 41.10%
λ=1e-03 | Sparsity: 99.97% | Accuracy: 33.43%
λ=2e-03 | Sparsity: 100.00% | Accuracy: 10.00%


## Done!
All plots saved in `output/` directory. Results summary printed above.